In [59]:
import json
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display,update_display
from scraper1 import fetch_website_contents,fetch_website_links
from openai import OpenAI


In [3]:
openai = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama3.1"
)

MODEL = "llama3.1"

In [ ]:
links= fetch_website_links ("https://ttuhscep.edu/")
links

['https://dev.ttuhscep.edu/outage-updates/index.aspx',
 'https://securelb.imodules.com/s/1422/c21/form.aspx?sid=1422&gid=1004&pgid=5202&appealcode=ELP24OAFOXPF',
 '/campus_map/index.aspx',
 '/',
 'https://securelb.imodules.com/s/1422/c21/form.aspx?sid=1422&gid=1004&pgid=5202&appealcode=ELP24OAFOXPF',
 '/campus_map/index.aspx',
 '/campus-life/why-el-paso.aspx',
 'https://www.ttuhscepimpact.com/',
 '/academics/admissions-and-aid.aspx',
 '/campus-life/index.aspx',
 'https://banapps.texastech.edu/itis/onlinedirectory/home',
 '/human-resources/index.aspx#JOBS',
 'https://www.facebook.com/tthealthelpaso',
 'https://www.instagram.com/tthealthelpaso/',
 'https://www.linkedin.com/school/texas-tech-university-health-sciences-center-el-paso/',
 'https://www.youtube.com/user/TTUHSCElPaso',
 '/',
 '#',
 '/about/index.aspx',
 '/about/president/index.aspx',
 '/about/Mission.aspx',
 '/about/strategic-plan.aspx',
 '/about/president/leadership.aspx',
 '/about/support-departments.aspx',
 '/values-based-c

In [35]:
link_system_prompt= """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links":[
        {"type":"about-page","url":"https://www.ttuhscepimpact.com/"},
        {"type": "admissions-page",
            "url": "https://ttuhscep.edu/academics/admissions-and-aid.aspx"}
    ]
}
IMPORTANT:
- URLs must be complete absolute URLs.
- Every URL must start with https://
- Never return relative URLs such as /about/page.aspx
- Never return URLs such as about/page.aspx
- Do not include Privacy Policy or Terms of Service.
"""

In [36]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the Orgnization, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt +="\n".join(links)
    return user_prompt


In [37]:

print(get_links_user_prompt("https://ttuhscep.edu/"))


    Here is the list of links on the website https://ttuhscep.edu/ -
Please decide which of these are relevant web links for a brochure about the Orgnization, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy.

Links (some might be relative links):

https://dev.ttuhscep.edu/outage-updates/index.aspx
https://securelb.imodules.com/s/1422/c21/form.aspx?sid=1422&gid=1004&pgid=5202&appealcode=ELP24OAFOXPF
/campus_map/index.aspx
/
https://securelb.imodules.com/s/1422/c21/form.aspx?sid=1422&gid=1004&pgid=5202&appealcode=ELP24OAFOXPF
/campus_map/index.aspx
/campus-life/why-el-paso.aspx
https://www.ttuhscepimpact.com/
/academics/admissions-and-aid.aspx
/campus-life/index.aspx
https://banapps.texastech.edu/itis/onlinedirectory/home
/human-resources/index.aspx#JOBS
https://www.facebook.com/tthealthelpaso
https://www.instagram.com/tthealthelpaso/
https://www.linkedin.com/school/texas-tech-university-health-sciences-center-el-paso/
https://www.youtube.com/us

In [30]:
from urllib.parse import urljoin

def select_relevant_links(url):
    response =openai.chat.completions.create(
        model= MODEL,
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_prompt(url)}
        ],
        response_format= {"type":"json_object"}
    )
    
    result= response.choices[0].message.content
    links= json.loads(result)
    for link in links["links"]:
        link["url"] = urljoin(url, link["url"])

    return links

In [31]:
select_relevant_links("https://ttuhscep.edu/")

{'links': [{'type': 'about-page',
   'url': 'https://www.ttuhscep.edu/about/index.aspx'},
  {'type': 'company-page',
   'url': 'https://www.ttuhscep.edu/academics/index.aspx'},
  {'type': 'careers-page',
   'url': 'https://www.ttuhscep.edu/human-resources/index.aspx#JOBS'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/tthealthelpaso'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/school/texas-tech-university-health-sciences-center-el-paso/'},
  {'type': 'youtube', 'url': 'https://www.youtube.com/user/TTUHSCElPaso'},
  {'type': 'instagram', 'url': 'https://www.instagram.com/tthealthelpaso/'},
  {'type': 'twitter', 'url': 'https://x.com/TTHealthElPaso'},
  {'type': 'news', 'url': 'https://www.ttuhscepimpact.com/'},
  {'type': 'news',
   'url': 'https://www.ttuhscepimpact.org/',
   'alturl': 'https://www.ttuhscepimpact.com/'},
  {'type': 'news', 'url': 'https://www.ttuhscepimpact.org/border-health-fund'},
  {'type': 'admissions-page',
   'url': 'https://www.ttuhsc

In [23]:
select_relevant_links("https://huggingface.co/")

{'links': [{'type': 'home-page', 'url': 'https://huggingface.co/'},
  {'type': 'models-page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets-page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces-page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'storage-page', 'url': 'https://huggingface.co/storage'},
  {'type': 'docs-page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn-page', 'url': 'https://huggingface.co/learn'},
  {'type': 'join-page', 'url': 'https://huggingface.co/join'},
  {'type': 'blog-page', 'url': 'https://huggingface.co/blog'},
  {'type': 'organization-page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog-post-page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github-page', 'url': 'https://github.com/huggingface'}]}

In [33]:
## SECOND STEP make the brochure

def fetch_page_and_all_relevant_links(url):
    contents= fetch_website_contents(url)
    relevant_links= select_relevant_links(url)
    result= f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link:{link['type']}\n"
        result += fetch_website_contents(link["url"])
        
    return result

In [39]:
print(fetch_page_and_all_relevant_links("https://huggingface.co/"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-27B
Updated
4 days ago
•
666k
•
11k
unsloth/Qwen3.8-27B-GGUF
Updated
3 days ago
•
3.56M
•
1.78k
Qwen/Qwen3.8-2.4T-A95B
Updated
6 days ago
•
11.2k
•
1.06k
Lightricks/LTX-2.5
Updated
1 day ago
•
504k
•
1.2k
MiniMaxAI/MiniMax-Music3
Updated
4 days ago
•
11.7k
•
939
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
165
MiniMax Music 3 Studio
🎵
165
Generate custom m

In [40]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [43]:
def get_brochure_user_prompt(company_name, url):

    user_prompt = f"""
You are looking at a company called: {company_name}

Here are the contents of its landing page and other relevant pages.

Use this information to build a short brochure of the company
in markdown without code blocks.

"""

    print("Starting website scraping...")
    print("URL:", repr(url))

    website_content = fetch_page_and_all_relevant_links(url)

    user_prompt += website_content

    user_prompt = user_prompt[:5000]

    return user_prompt

In [44]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Starting website scraping...
URL: 'https://huggingface.co'


'\nYou are looking at a company called: HuggingFace\n\nHere are the contents of its landing page and other relevant pages.\n\nUse this information to build a short brochure of the company\nin markdown without code blocks.\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n4 days ago\n•\n666k\n•\n11k\nunsloth/Qwen3.8-27B-GGUF\nUpdated\n3 days ago\n•\n3.56M\n

In [45]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3.1",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [47]:
create_brochure("HuggingFace", "https://huggingface.co")

Starting website scraping...
URL: 'https://huggingface.co'


**Hugging Face: Empowering the Machine Learning Community**

Welcome to Hugging Face, the leading collaboration platform for machine learning (ML) enthusiasts and professionals. Our mission is to accelerate the pace of innovation in AI research and development by providing a unified platform for creating, discovering, and sharing ML models, datasets, and applications.

**The Power of Collaboration**

At Hugging Face, we believe that the future of AI is built on collaboration and openness. Our platform provides a rich ecosystem for developers to come together, share ideas, and contribute to the growth of the ML community. With our open-source model library, you can browse over 2 million models and access a vast collection of datasets, applications, and tools.

**What We Offer**

* **Collaboration Platform**: Host and collaborate on public models, datasets, and applications, making it easier to build and deploy ML projects.
* **Open-Source Model Library**: Browse over 2 million models and access a vast collection of datasets, applications, and tools.
* **Machine Learning Ecosystem**: Explore AI applications, discover new techniques, and stay up-to-date with the latest innovations.
* **Community Support**: Join our vibrant community of developers, researchers, and ML enthusiasts, and participate in discussions, share knowledge, and learn from one another.

**Our Team**

Our team of experienced professionals is dedicated to building a platform that empowers the ML community. We believe in the importance of collaboration, innovation, and community-driven growth.

**Careers**

Are you passionate about machine learning and collaboration? We're always looking for talented individuals to join our team. Check out our [Careers page](link) to explore our current openings and learn more about our company culture.

**Stay Connected**

Follow us on [Twitter](link) and [LinkedIn](link) to stay up-to-date with the latest news, research, and innovations in the machine learning community.

Join the Hugging Face community today and start building a better future for AI together!

In [54]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3.1",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [61]:
stream_brochure("HuggingFace", "https://huggingface.co")

Starting website scraping...
URL: 'https://huggingface.co'


**Hugging Face Brochure**

**Welcome to the Future of AI**

Hugging Face is the AI community building the future. Our platform is where the machine learning community collaborates on models, datasets, and applications.

**The Home of Machine Learning**

Create, discover and collaborate on ML better. Host and collaborate on unlimited public models, datasets, and applications. Move faster with the HF Open source strategy.

**Explore Our Features**

* Browse 2M+ models, including trending models, latest updates, and popular ones
* Explore AI Apps and discover new possibilities
* Browse 500k+ datasets, including latest updates and popular ones
* Host and collaborate on unlimited public models, datasets, and applications
* Move faster with the HF Open source strategy

**Community Building**

Hugging Face is the collaboration platform for the machine learning community. Our platform is where researchers, developers, and enterprises come together to share knowledge, best practices, and innovations.

**Company Culture**

At Hugging Face, we believe in open-source, transparency, and collaboration. We are committed to building a community that is inclusive, diverse, and open to all.

**Career Opportunities**

Join our team of passionate and talented individuals who are shaping the future of AI. We are always looking for skilled professionals to join our ranks.

**Pricing**

Our pricing plans are flexible and designed to suit your needs. From Enterprise Support to Inference Providers, we have solutions for all.

**Get Started**

Sign up today and start exploring our platform. Discover new models, datasets, and applications. Join our community and start collaborating.

**Learn More**

* Visit our blog for the latest news and updates
* Explore our documentation for more information on our platform
* Join our community on Discord, Forum, and GitHub

**Contact Us**

If you have any questions or need help, don't hesitate to contact us. We are always here to support you.

**Hugging Face - Building the Future of AI**

In [62]:
def answer_question(question, url):
    website_content = fetch_page_and_all_relevant_links(url)

    messages = [
        {
            "role": "system",
            "content": """
            You are a helpful assistant.
            Answer the user's question using only the
            website information provided.
            If the information is not available, say so.
            """
        },
        {
            "role": "user",
            "content": f"""
            User question:
            {question}

            Website information:
            {website_content}
            """
        }
    ]

    response = openai.chat.completions.create(
        model="llama3.1",
        messages=messages
    )

    return response.choices[0].message.content

In [64]:
question = input("What would you like to know? ")

answer = answer_question(
    question,
    "https://huggingface.co"
)

display(Markdown(answer))

The text model mentioned in your question is not explicitly mentioned in the provided website information. However, there are text generation models mentioned such as Lightricks/LTX-2.5, Qwen/Qwen3.8-27B, and Lightricks/LTX-2.5.